# CARE Results Analysis & Visualization
Load experiment results from Drive and generate comprehensive plots.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/CARE_Experiments'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

## Load All Results

In [ ]:
# Load main results
results_df = pd.read_csv(f'{DRIVE_ROOT}/all_results.csv')
print(f'Loaded {len(results_df)} experiments')
results_df

In [ ]:
# Load per-epoch metrics for each experiment
all_metrics = {}
results_path = Path(DRIVE_ROOT) / 'results'

for exp_dir in results_path.iterdir():
    if exp_dir.is_dir():
        metrics_file = exp_dir / 'neuron_metrics.csv'
        if metrics_file.exists():
            all_metrics[exp_dir.name] = pd.read_csv(metrics_file)
            print(f'Loaded: {exp_dir.name}')

print(f'\nTotal: {len(all_metrics)} experiments with detailed metrics')

## 1. Accuracy Comparison (Control vs CARE)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

datasets = ['fashion_mnist', 'cifar10', 'nmnist']
for i, dataset in enumerate(datasets):
    ax = axes[i]
    subset = results_df[results_df['dataset'] == dataset]
    if subset.empty:
        ax.set_title(f'{dataset} (no data)')
        continue
    
    depths = sorted(subset['depth'].unique())
    x = np.arange(len(depths))
    width = 0.35
    
    control_acc = [subset[(subset['depth']==d) & (subset['mode']=='control')]['best_accuracy'].values[0] 
                   if len(subset[(subset['depth']==d) & (subset['mode']=='control')]) > 0 else 0 
                   for d in depths]
    care_acc = [subset[(subset['depth']==d) & (subset['mode']=='care')]['best_accuracy'].values[0] 
                if len(subset[(subset['depth']==d) & (subset['mode']=='care')]) > 0 else 0 
                for d in depths]
    
    bars1 = ax.bar(x - width/2, control_acc, width, label='Control', color='#e74c3c', alpha=0.8)
    bars2 = ax.bar(x + width/2, care_acc, width, label='CARE', color='#27ae60', alpha=0.8)
    
    ax.set_xlabel('ResNet Depth', fontsize=12)
    ax.set_ylabel('Best Accuracy', fontsize=12)
    ax.set_title(dataset.replace('_', ' ').upper(), fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'ResNet-{d}' for d in depths])
    ax.legend()
    ax.set_ylim(0, 1)
    
    # Add value labels
    for bar, val in zip(bars1, control_acc):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{val:.1%}', ha='center', va='bottom', fontsize=8)
    for bar, val in zip(bars2, care_acc):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{val:.1%}', ha='center', va='bottom', fontsize=8)

plt.suptitle('CARE vs Control: Accuracy Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/plots/accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Dead Neuron Ratio Over Training

In [ ]:
# Plot dead neuron curves for each dataset
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, dataset in enumerate(datasets):
    ax = axes[i]
    
    for exp_name, metrics in all_metrics.items():
        if dataset not in exp_name:
            continue
        
        mode = 'CARE' if 'care' in exp_name else 'Control'
        depth = int(exp_name.split('resnet')[1].split('_')[0])
        color = '#27ae60' if 'care' in exp_name else '#e74c3c'
        linestyle = '-' if 'care' in exp_name else '--'
        
        if 'global_dead_ratio' in metrics.columns:
            ax.plot(metrics['epoch'], metrics['global_dead_ratio'], 
                    label=f'{mode} (d={depth})', color=color, linestyle=linestyle, alpha=0.8)
    
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Dead Neuron Ratio', fontsize=12)
    ax.set_title(dataset.replace('_', ' ').upper(), fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_ylim(0, 1)

plt.suptitle('Dead Neuron Ratio During Training', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/plots/dead_neuron_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Revival Events Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, dataset in enumerate(datasets):
    ax = axes[i]
    
    for exp_name, metrics in all_metrics.items():
        if dataset not in exp_name:
            continue
        
        mode = 'CARE' if 'care' in exp_name else 'Control'
        color = '#27ae60' if 'care' in exp_name else '#e74c3c'
        
        if 'global_revived' in metrics.columns:
            cumulative_revived = metrics['global_revived'].cumsum()
            ax.plot(metrics['epoch'], cumulative_revived, 
                    label=f'{mode}', color=color, linewidth=2)
    
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Cumulative Revived Neurons', fontsize=12)
    ax.set_title(dataset.replace('_', ' ').upper(), fontsize=14, fontweight='bold')
    ax.legend()

plt.suptitle('Neuron Revival Events (Dead → Active)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/plots/revival_events.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Depth Scaling Analysis

In [ ]:
# Compute CARE benefit (CARE acc - Control acc) vs depth
fig, ax = plt.subplots(figsize=(10, 6))

for dataset in datasets:
    subset = results_df[results_df['dataset'] == dataset]
    if subset.empty:
        continue
    
    depths = sorted(subset['depth'].unique())
    benefits = []
    
    for d in depths:
        control = subset[(subset['depth']==d) & (subset['mode']=='control')]['best_accuracy']
        care = subset[(subset['depth']==d) & (subset['mode']=='care')]['best_accuracy']
        if len(control) > 0 and len(care) > 0:
            benefit = care.values[0] - control.values[0]
            benefits.append(benefit * 100)  # Convert to percentage points
        else:
            benefits.append(0)
    
    ax.plot(depths, benefits, 'o-', label=dataset.replace('_', ' ').title(), linewidth=2, markersize=8)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Network Depth (ResNet-X)', fontsize=12)
ax.set_ylabel('CARE Benefit (% points)', fontsize=12)
ax.set_title('CARE Benefit Increases with Depth?', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/plots/depth_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Summary Table

In [ ]:
# Create summary comparison table
summary_data = []

for dataset in datasets:
    subset = results_df[results_df['dataset'] == dataset]
    for depth in sorted(subset['depth'].unique()):
        control = subset[(subset['depth']==depth) & (subset['mode']=='control')]
        care = subset[(subset['depth']==depth) & (subset['mode']=='care')]
        
        if len(control) > 0 and len(care) > 0:
            c_acc = control['best_accuracy'].values[0]
            h_acc = care['best_accuracy'].values[0]
            c_dead = control['final_dead_ratio'].values[0]
            h_dead = care['final_dead_ratio'].values[0]
            
            summary_data.append({
                'Dataset': dataset,
                'Depth': depth,
                'Control Acc': f'{c_acc:.1%}',
                'CARE Acc': f'{h_acc:.1%}',
                'Δ Accuracy': f'{(h_acc-c_acc)*100:+.1f}%',
                'Control Dead': f'{c_dead:.1%}',
                'CARE Dead': f'{h_dead:.1%}',
                'Winner': 'CARE' if h_acc > c_acc else 'Control' if c_acc > h_acc else 'Tie'
            })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_markdown(index=False))
summary_df.to_csv(f'{DRIVE_ROOT}/summary_table.csv', index=False)

## 6. Export Report

In [ ]:
# Generate markdown report
report = f"""
# CARE Rigorous Evaluation Report

## Configuration
- Epochs: 30
- Batch Size: 64
- Learning Rate: 1e-3
- Random Seed: 42

## Results Summary

{summary_df.to_markdown(index=False)}

## Key Findings

1. **CARE Benefit by Dataset**: [To be filled based on results]
2. **Depth Scaling Effect**: [To be filled based on results]
3. **Dead Neuron Reduction**: [To be filled based on results]

## Plots

See `plots/` directory for visualizations.
"""

with open(f'{DRIVE_ROOT}/REPORT.md', 'w') as f:
    f.write(report)

print(f'Report saved to: {DRIVE_ROOT}/REPORT.md')